<a href="https://colab.research.google.com/github/elliemci/agents/blob/main/monitoring_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Observability and Evaluation of Agents

Tracking internal operations through logs, metrics and spancs to observe agent inner workings

## Required Libraries

In [ ]:
%pip install 'smolagents[telemetry]'
%pip install opentelemetry-sdk opentelemetry-exporter-otlp openinference-instrumentation-smolagents
%pip install langfuse datasets 'smolagents[gradio]'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/ColabNotebooks/AgentsCourse

Mounted at /content/drive
/content/drive/MyDrive/ColabNotebooks/AgentsCourse


## Set Environmental variables and Connection to Langfuse OpenTelemetry

In [ ]:
import os
import base64
from google.colab import userdata


langfuse_public_key = userdata.get('LANGFUSE_PUBLIC_KEY')
langfuse_secret_key = userdata.get('LANGFUSE_SECRET_KEY')

os.environ['LANGFUSE_PUBLIC_KEY'] = langfuse_public_key
os.environ['LANGFUSE_SECRET_KEY'] = langfuse_secret_key
os.environ["LANGFUSE_HOST"] = "https://us.cloud.langfuse.com"

# Langfuse authentication: langfuse public and private keys are combined into
# the format public_key:secret_key and then encoded using encode() to prepare for
# base64 encoding which reoresents binary data in text format safe to transmit,
# the result of nase64 encoding is decoded back into a string
LANGFUSE_AUTH = base64.b64encode(
    f"{os.environ.get('LANGFUSE_PUBLIC_KEY')}:{os.environ.get('LANGFUSE_SECRET_KEY')}".encode()
).decode()

# set LangFuse OpenTelemetry endpoint URL environmental vareiable
# for sending OpenTelemetry logs, metrics, traces to Langfuse
os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = os.environ.get("LANGFUSE_HOST") + "/api/public/otel"
# environment variable to configure HTTP headers for the OpenTelemetry data export;
# to ensure that the data is sent with proper authentication credentials the Autorization
# header is set to Basic and followed by the LANGFUSE_AUTh value calculated earlier
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"Authorization=Basic {LANGFUSE_AUTH}"

hugging_fase_token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = hugging_fase_token

## Setup a Trace-provider

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from openinference.instrumentation.smolagents import SmolagentsInstrumentor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.trace.export import SimpleSpanProcessor


# create a TracerProvider for OpenTelemetry
trace_provider = TracerProvider()

# add a SimpleSpanProcessor with the OTLPSpanExporter to send traces
trace_provider.add_span_processor(SimpleSpanProcessor(OTLPSpanExporter()))

# set the global default tracer provider
trace.set_tracer_provider(trace_provider)
tracer = trace.get_tracer(__name__)

# Instrument smolagents with the configured provider
SmolagentsInstrumentor().instrument(tracer_provider=trace_provider)

## Test Instrumentation

Test the set up running a simple CodeAgent smolagents, should be able to see logs/spans in observability dashboard https://us.cloud.langfuse.com/ sign in with Google.

In [ ]:
from smolagents import HfApiModel, CodeAgent

# instantiate a simple agent to test instrumentation
agent = CodeAgent(
    tools=[],
    model=HfApiModel()
)

request = "Write a short poem about the beauty of nature."

agent.run(request)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Write a short poem about the beauty of nature.                                                                  │
│                                                                                                                 │
╰─ HfApiModel - Qwen/Qwen2.5-Coder-32B-Instruct ──────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Define the structure and content of the poem                                                                   
  poem_lines = [                                                                                                   
      "In mountains high where eagles dare to soar,",                                                              
      "Nature paints the sky with hues of blue;",                                                                  
      "Sunset's blush on rivers, quiet and pure,",                                                                 
      "Reflects a beauty, tranquil and renew.",                                                                    
      "",                                                                                                          
      "Whispers of the breeze through ancient pines,",                                                             
      "Symphony of leaves in dance divine;",                                                                       
      "Wildflowers bloom in fields of vibrant green,",                                                             
      "Life's celebration beneath the zenith's keen.",                                                             
      "",                                                                                                          
      "Stars at night, in heavens they align,",                                                                    
      "Guiding us with gentle, lighted sign;",                                                                     
      "Nature's symphonic drama, never still,",                                                                    
      "A world of wonder, vast, and full and thrill."                                                              
  ]                                                                                                                
                                                                                                                   
  # Join the lines to form the complete poem                                                                       
  poem = "\n".join(poem_lines)                                                                                     
                                                                                                                   
  print(poem)                                                                                                      
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
In mountains high where eagles dare to soar,
Nature paints the sky with hues of blue;
Sunset's blush on rivers, quiet and pure,
Reflects a beauty, tranquil and renew.

Whispers of the breeze through ancient pines,
Symphony of leaves in dance divine;
Wildflowers bloom in fields of vibrant green,
Life's celebration beneath the zenith's keen.

Stars at night, in heavens they align,
Guiding us with gentle, lighted sign;
Nature's symphonic drama, never still,
A world of wonder, vast, and full and thrill.

Out: None

[Step 1: Duration 11.21 seconds| Input tokens: 2,019 | Output tokens: 234]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("In mountains high where eagles dare to soar,\nNature paints the sky with hues of blue;\nSunset's   
  blush on rivers, quiet and pure,\nReflects a beauty, tranquil and renew.\n\nWhispers of the breeze through       
  ancient pines,\nSymphony of leaves in dance divine;\nWildflowers bloom in fields of vibrant green,\nLife's       
  celebration beneath the zenith's keen.\n\nStars at night, in heavens they align,\nGuiding us with gentle,        
  lighted sign;\nNature's symphonic drama, never still,\nA world of wonder, vast, and full and thrill.")           
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: In mountains high where eagles dare to soar,
Nature paints the sky with hues of blue;
Sunset's blush on rivers, quiet and pure,
Reflects a beauty, tranquil and renew.

Whispers of the breeze through ancient pines,
Symphony of leaves in dance divine;
Wildflowers bloom in fields of vibrant green,
Life's celebration beneath the zenith's keen.

Stars at night, in heavens they align,
Guiding us with gentle, lighted sign;
Nature's symphonic drama, never still,
A world of wonder, vast, and full and thrill.

[Step 2: Duration 8.85 seconds| Input tokens: 4,673 | Output tokens: 402]

"In mountains high where eagles dare to soar,\nNature paints the sky with hues of blue;\nSunset's blush on rivers, quiet and pure,\nReflects a beauty, tranquil and renew.\n\nWhispers of the breeze through ancient pines,\nSymphony of leaves in dance divine;\nWildflowers bloom in fields of vibrant green,\nLife's celebration beneath the zenith's keen.\n\nStars at night, in heavens they align,\nGuiding us with gentle, lighted sign;\nNature's symphonic drama, never still,\nA world of wonder, vast, and full and thrill."

## Observe and Evluate

The observability tool recod a trace that contains the spans and sub-spans, each step if the agenti's logic:<br>
* The tool calls DudckDuckGoSearchTool
* The LLM calls

In [ ]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, HfApiModel

search_tool = DuckDuckGoSearchTool()
agent = CodeAgent(tools=[search_tool], model=HfApiModel())

agent.run("What is the forecast for AT&T stock price?")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the forecast for AT&T stock price?                                                                      │
│                                                                                                                 │
╰─ HfApiModel - Qwen/Qwen2.5-Coder-32B-Instruct ──────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  stock_price_info = web_search(query="AT&T stock price forecast 2023")                                            
  print(stock_price_info)                                                                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results

[AT&T Inc. (T) Stock Forecast & Price Targets - Stock Analysis](https://stockanalysis.com/stocks/t/forecast/)
Stock Price Forecast The 17 analysts with 12-month price forecasts for AT&T stock have an average target of 26.94, 
with a low estimate of 18 and a high estimate of 32. The average target predicts an increase of 2.08% from the 
current stock price of 26.39.

[AT&T (T) Stock Forecast and Price Target 2025 - MarketBeat](https://www.marketbeat.com/stocks/NYSE/T/forecast/)
Learn why top analysts are making this stock forecast for AT&T at MarketBeat. ... 6/5/2023: HSBC Subscribe to 
MarketBeat All Access for the recommendation accuracy rating : Lower Target: ... the average twelve-month stock 
price forecast for AT&T is $27.24, with a high forecast of $32.00 and a low forecast of $18.00. ...

[T Stock Quote Price and Forecast | CNN](https://www.cnn.com/markets/stocks/T)
View AT&T Inc T stock quote prices, financial information, real-time forecasts, and company news from CNN. ... 2023
4:25pm ET AT&T (T) ... 1-year stock price forecast T Competitors. Facts Insights ...

[AT&T (T) Stock Price, News & Analysis - MarketBeat](https://www.marketbeat.com/stocks/NYSE/T/)
Read more about AT&T's stock forecast and price target. Earnings and Valuation 1.9 / 5 Proj. Earnings Growth 6.07%.
Earnings Growth. Earnings for AT&T are expected to grow by 6.07% in the coming year, from $2.14 to $2.27 per share.
Price to Earnings Ratio vs. the Market.

[AT&T (T) stock Forecast for 2023, 2024, 2025, 2026 - 
leoprophet.com](https://leoprophet.com/stock_forecasts/forecast_t/)
AT&T stock forecast for 05.10.2023. Estimated Average Forecasted AT&T Price: 13.64 Positive intraday dynamics of 
the instrument is expected with 1.902% volatility is expected. Pessimistic forecast: 13.51 Optimistic: 13.77 AT&T 
stock forecast for 06.10.2023.

[AT&T Stock Forecast For 2023: What To Watch For 
(NYSE:T)](https://seekingalpha.com/article/4560843-att-stock-forecast-2023)
AT&T's shares have room for further upside, with my 2023 target price forecast pointing to a +21% capital 
appreciation potential. Read the forecast here. ... AT&T's stock price has gone up by +11. ...

[AT&T (NYSE:T) Stock Forecast & Analyst Predictions - Simply Wall 
St](https://simplywall.st/stocks/us/telecom/nyse-t/att/future)
Discover AT&T's earnings and revenue growth rates, forecasts, and the latest analyst predictions while comparing 
them to its industry peers. ... AT&T is forecast to grow earnings and revenue by 11.3% and 1.7% per annum 
respectively. EPS is expected to grow by 12.3% per annum. ... Stock prices; Dividends, Splits and Actions; ICE 
Market Data; SEC ...

[Forecasting The Future: 13 Analyst Projections For 
AT&T](https://www.nasdaq.com/articles/forecasting-future-13-analyst-projections-att)
In the assessment of 12-month price targets, analysts unveil insights for AT&T, presenting an average target of 
$22.23, a high estimate of $30.00, and a low estimate of $18.00.

[AT&T Inc Stock Forecast, Predictions & Price Target - 
WallStreetZen](https://www.wallstreetzen.com/stocks/us/nyse/t/stock-forecast)
The average AT&T stock price prediction forecasts a potential upside of 5.08% from the current T share price of 
$26.83. What is T's Earnings Per Share (EPS) forecast for 2025-2027? (NYSE: T) AT&T's current Earnings Per Share 
(EPS) is $1.49. On average, analysts forecast that T's EPS will be $2.10 for 2025, with the lowest EPS forecast at 
$2.01 ...

[AT&T (T) stock forecast for 2023. Forecast tables and 
graphs.](https://leoprophet.com/stock_forecasts/forecast_t/for2023/)
Forecasts are adjusted once a day taking into account the price change of the previous day. To date, analysts have 
a $17.25 target price for AT&T stock stock. Today 200 Day Moving Average is the resistance level (17.06 $). 50 Day 
Moving Average is the support level (14.55 $).

Out: None

[Step 1: Duration 5.45 seconds| Input tokens: 2,080 | Output tokens: 86]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(average_forecast)                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: 25.81

[Step 3: Duration 3.72 seconds| Input tokens: 9,156 | Output tokens: 271]

25.81

## Online Evaluation

### Production Metrics

1. Cost
2. Latency
3. Attributes or custom tags
4. User Feedback
5. LLM-as-a-Judge

Additional attributes like user ID, session IDs ot tags can be passed when evaluation

In [ ]:
alpha_vantage_api_key = userdata.get('ALPHA_VANTAGE_STOCK_API_KEY')
os.environ['ALPHA_VANTAGE_STOCK_API_KEY'] = alpha_vantage_api_key

In [ ]:
with tracer.start_as_current_span("Smolagent-Trace") as span:
    span.set_attribute("langfuse.user.id", "smolagent-user-123")
    span.set_attribute("langfuse.session.id", "smolagent-session-123456789")
    span.set_attribute("langfuse.tags", ["Sock Market-question", "testing-agents"])

    agent.run("Use the alpha_vantage_api_key code varible and Alpha Vantage stock API to find a forecast for MSTY's stock price and the divident?")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Use the alpha_vantage_api_key code varible and Alpha Vantage stock API to find a forecast for MSTY's stock      │
│ price and the divident?                                                                                         │
│                                                                                                                 │
╰─ HfApiModel - Qwen/Qwen2.5-Coder-32B-Instruct ──────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import requests                                                                                                  
  import pandas as pd                                                                                              
                                                                                                                   
  # Fetching stock data for MSTY                                                                                   
  url =                                                                                                            
  f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=MSTY&apikey={alpha_vantage_api_key}'       
  response = requests.get(url)                                                                                     
  data = response.json()                                                                                           
  print("Stock data fetched:", data)                                                                               
                                                                                                                   
  # Fetching dividend data for MSTY                                                                                
  dividend_url =                                                                                                   
  f'https://www.alphavantage.co/query?function=TIME_SERIES_MONTHLY_ADJUSTED&symbol=MSTY&apikey={alpha_vantage_api  
  _key}'                                                                                                           
  dividend_response = requests.get(dividend_url)                                                                   
  dividend_data = dividend_response.json()                                                                         
  print("Dividend data fetched:", dividend_data)                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Warning to user: Code execution failed due to an unauthorized import - Consider passing said import under 
`additional_authorized_imports` when initializing your CodeAgent.

Code execution failed at line 'import requests' due to: InterpreterError: Import of requests is not allowed. 
Authorized imports are: ['itertools', 'queue', 'random', 'datetime', 'statistics', 'unicodedata', 'collections', 
'time', 'stat', 're', 'math'\]

...

[Step 10: Duration 22.52 seconds| Input tokens: 83,148 | Output tokens: 3,212]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 11 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(f"The forecasted stock price of MSTY using SMA is approximately 19.62. The latest and next          
  dividend per share is $1.34.")                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: The forecasted stock price of MSTY using SMA is approximately 19.62. The latest and next 
dividend per share is $1.34.

[Step 11: Duration 9.59 seconds| Input tokens: 99,042 | Output tokens: 3,381]

A user feedback mechanism for recording User Feedback for a Gradio Chatbot integrated with Langfuse; check out Scores under Tracing on Langfuse dashboard

In [ ]:
import gradio as gr

from opentelemetry.trace import format_trace_id # for tracing the execution and generating trace ids
from langfuse import Langfuse # for sending trace data and user feedback to Langfuse

langfuse = Langfuse()
model = HfApiModel()
agent = CodeAgent(tools=[search_tool], model=model, add_base_tools=True)

# capture of the OpenTelemetry trace id and attach a score to the trace
# store the current trace_id globally, use to link user feedback to traces
formatted_trace_id = None

def respond(prompt, history):
  """ The respond function executes every time the user submits a prompt."""

  # start a new trace span
  with trace.get_tracer(__name__).start_as_current_span("Smolagent-Trace") as span:

      # run the agent with the user's prompt
      output = agent.run(prompt)

      current_span = trace.get_current_span()
      span_context = current_span.get_span_context()
      trace_id = span_context.trace_id

      global formatted_trace_id

      # store the trace id
      formatted_trace_id = str(format_trace_id(trace_id))

      langfuse.trace(id=formatted_trace_id, input=prompt, output=output)

  # append agent's reponse to the history
  history.append({
        "role": "assistant",
        "content": str(output)})

  return history

def handle_like(data: gr.LikeData):
  """ The function is called when the user clicks on like or dislike button."""
  # map user feedback to a 1=like or 0=dislike
  if data.liked:
    # send a score to Langfuse using the formatted_trace_id to associate the feedback with the correct trace
      langfuse.score(
          value=1,
          name="user-feedback",
          trace_id=formatted_trace_id
      )
  else:
      langfuse.score(
          value=0,
          name="user-feedback",
          trace_id=formatted_trace_id
      )

# Launch the Chatbot with Gradio interface
with gr.Blocks() as demo:
    chatbot = gr.Chatbot(label="Chat", type="messages")
    prompt_box = gr.Textbox(placeholder="Type your message...", label="Your message")

    # when the user presses 'Enter' on the prompt, run 'respond'
    prompt_box.submit(
        fn=respond,
        inputs=[prompt_box, chatbot],
        outputs=chatbot
    )

    # set up an event handler to call the respond funcion handle_like
    # when the user clicks a 'like' button on a message
    chatbot.like(handle_like, None, None)

demo.launch()


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f4bca6ebefa7cab83d.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
langfuse.score(name="user-feedback", value=1)

LLM-as-a-judge - separate LLM call to evaluate the model's output; create an evalutation templet at Langfuse dashboard under Evaluation

## Offline Evaluation

Run agent on a benchmark dataset with prompt and expected output pairs and link the trace to the dataset item,compare outputs to the expected results.

### Load a banchmark dataset

In [ ]:
import pandas as pd
from datasets import load_dataset

# use GSM8K from Hugging Face as a benchmark set which contains math questions andtheir solutions
dataset = load_dataset("openai/gsm8k", 'main', split='train')
df = pd.DataFrame(dataset)
print("First few rows of GSM8K dataset:")
print(df.head())

First few rows of GSM8K dataset:
                                            question  \
0  Natalia sold clips to 48 of her friends in Apr...   
1  Weng earns $12 an hour for babysitting. Yester...   
2  Betty is saving money for a new wallet which c...   
3  Julie is reading a 120-page book. Yesterday, s...   
4  James writes a 3-page letter to 2 different fr...   

                                              answer  
0  Natalia sold 48/2 = <<48/2=24>>24 clips in May...  
1  Weng earns 12/60 = $<<12/60=0.2>>0.2 per minut...  
2  In the beginning, Betty has only 100 / 2 = $<<...  
3  Maila read 12 x 2 = <<12*2=24>>24 pages today....  
4  He writes each friend 3*2=<<3*2=6>>6 pages a w...  


Create a dataset entity to track the runs in Langfuse and add each item from the dataset to the system. After uploading the benchmark dataset in Langfuse, check out Dataset, Items, Inputs and Expected Outputs at the Langfuse dashboard

In [ ]:
from langfuse import Langfuse
langfuse = Langfuse()

langfuse_dataset_name = "gsm8k_dataset_huggingface"

# create a dataset in Langfuse
langfuse.create_dataset(
    name=langfuse_dataset_name,
    description="GSM8K benchmark dataset uploaded from Huggingface",
    metadata={
        "date": "2025-04-13",
        "type": "benchmark"
    }
)

Dataset(id='cm9gm99c401j8ad07pzspxttx', name='gsm8k_dataset_huggingface', description='GSM8K benchmark dataset uploaded from Huggingface', metadata={'date': '2025-04-13', 'type': 'benchmark'}, project_id='cm7s3y7c8038bad07lfm2uve1', created_at=datetime.datetime(2025, 4, 14, 5, 11, 33, 509000, tzinfo=datetime.timezone.utc), updated_at=datetime.datetime(2025, 4, 14, 5, 11, 33, 509000, tzinfo=datetime.timezone.utc))

In [ ]:
for idx, row in df.iterrows():
    langfuse.create_dataset_item(
        dataset_name=langfuse_dataset_name,
        input={"text": row["question"]},
        expected_output={"text": row["answer"]},
        metadata={"source_index": idx}
    )
    # upload the first 10 items
    if idx >= 9:
        break

Compare Runs at Langfuse under Inpput, Output and Expected Output for Dataset Item of a
smolagent-notebook-run-01



In [ ]:
from opentelemetry.trace import format_trace_id

model = HfApiModel()

agent = CodeAgent(
    tools=[],
    model=model,
    add_base_tools=True
)

def run_smolagent(question):
  """ Function that runs agent with a given question and records the execution in Langfuse."""

  # start a trace spam
  with tracer.start_as_current_span("Smolagent-Trace") as span:
      # add a tag to the trace for easier identification
      span.set_attribute("langfuse.tag", "dataset-run")
      output = agent.run(question)
      # retrive current trace id
      current_span = trace.get_current_span()
      span_context = current_span.get_span_context()
      trace_id = span_context.trace_id
      # format_trace_id function is used to format the trace id for langfuse
      formatted_trace_id = format_trace_id(trace_id)

      # record trace in langfuse
      langfuse_trace = langfuse.trace(
          id=formatted_trace_id,
          input=question,
          output=output
        )
  return langfuse_trace, output

# run the evaluation with banchmark dataset
dataset = langfuse.get_dataset(langfuse_dataset_name)

# run agent against each dataset item, limited to first 10
for item in dataset.items:
    langfuse_trace, output = run_smolagent(item.input["text"])

    # link the trace to the dataset item for analysis
    item.link(
        langfuse_trace,
        run_name="smolagent-notebook-run-01",
        run_metadata={ "model": model.model_id }
    )

    # score the trace and add an evaluation score
    langfuse_trace.score(
        name="<example_eval>",
        value=1,
        comment="This is a comment"
    )

# flush data to ensure all telemetry is sent
langfuse.flush()

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Tina makes $18.00 an hour.  If she works more than 8 hours per shift, she is eligible for overtime, which is    │
│ paid by your hourly wage + 1/2 your hourly wage.  If she works 10 hours every day for 5 days, how much money    │
│ does she make?                                                                                                  │
│                                                                                                                 │
╰─ HfApiModel - Qwen/Qwen2.5-Coder-32B-Instruct ──────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  hourly_wage = 18.00                                                                                              
  regular_hours_per_day = 8                                                                                        
  overtime_hours_per_day = 2                                                                                       
  days_per_week = 5                                                                                                
                                                                                                                   
  # Regular pay calculation                                                                                        
  regular_pay_per_day = hourly_wage * regular_hours_per_day                                                        
  total_regular_pay = regular_pay_per_day * days_per_week                                                          
                                                                                                                   
  # Overtime pay calculation                                                                                       
  overtime_pay_rate = hourly_wage * 1.5                                                                            
  overtime_pay_per_day = overtime_pay_rate * overtime_hours_per_day                                                
  total_overtime_pay = overtime_pay_per_day * days_per_week                                                        
                                                                                                                   
  # Total pay calculation                                                                                          
  total_pay = total_regular_pay + total_overtime_pay                                                               
  print(total_pay)                                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
990.0

Out: None

[Step 1: Duration 27.92 seconds| Input tokens: 2,202 | Output tokens: 221]

...

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips   │
│ did Natalia sell altogether in April and May?                                                                   │
│                                                                                                                 │
╰─ HfApiModel - Qwen/Qwen2.5-Coder-32B-Instruct ──────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  clips_sold_april = 48                                                                                            
  clips_sold_may = clips_sold_april / 2                                                                            
  total_clips_sold = clips_sold_april + clips_sold_may                                                             
  print(total_clips_sold)                                                                                          
  final_answer(total_clips_sold)                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
72.0

Out - Final answer: 72.0

[Step 1: Duration 5.54 seconds| Input tokens: 2,173 | Output tokens: 115]